# Herramientas · Automatización

*Logging*, tareas programadas, PDF, Excel y entornos virtuales.

## 1. Nivel de *log* sin distinguir mayúsculas

El programa recibe el nivel de *log* por línea de comandos con `--log-level`. Modificarlo para que acepte `INFO` e `info` por igual, y escribir una función que use `logger.info` y `logger.debug` para mensajes de distinta importancia.

In [1]:
import argparse
import logging
import sys

parser = argparse.ArgumentParser(description="Nivel de verbosity")
# type=str.upper normaliza antes de comprobar choices
parser.add_argument("--log-level", type=str.upper, default="INFO", choices=["DEBUG", "INFO", "WARNING", "ERROR"])

def procesa(datos, logger):
    logger.info("Procesando %d elementos", len(datos))
    for d in datos:
        logger.debug("Elemento %r", d)
    return sum(datos)

for argumentos in (["--log-level=info"], ["--log-level=Debug"]):
    args = parser.parse_args(argumentos)
    logging.basicConfig(level=args.log_level, format="%(levelname)s - %(message)s", stream=sys.stdout, force=True)
    logger = logging.getLogger("logger")
    print(f"-- {argumentos[0]} --")
    procesa([1, 2, 3], logger)

-- --log-level=info --
INFO - Procesando 3 elementos
-- --log-level=Debug --
INFO - Procesando 3 elementos
DEBUG - Elemento 1
DEBUG - Elemento 2
DEBUG - Elemento 3


## 2. Tarea diaria con `schedule`

En lugar de ejecutar la tarea cada cierto intervalo, programarla todos los días a las 23:59. El bucle infinito `while True: schedule.run_pending()` es el que se usaría en un programa real; aquí se ejecuta unas pocas iteraciones.

In [2]:
import time
from datetime import datetime
import schedule

def tarea():
    print(f"{datetime.now():%H:%M:%S}: Tu tarea")

schedule.clear()
trabajo = schedule.every().day.at("23:59").do(tarea)
print("Próxima ejecución:", trabajo.next_run)
for _ in range(3):
    schedule.run_pending()
    time.sleep(0.1)
schedule.run_all()

Próxima ejecución: 2026-11-06 23:59:00
22:43:48: Tu tarea


## 3. PDF con `pypdf`

Escribir una función que elimine la última página de un PDF y otra que una todos los PDF de una carpeta.

In [3]:
import os
from pypdf import PdfReader, PdfWriter

os.makedirs("pdfs", exist_ok=True)
for nombre, paginas in (("a.pdf", 2), ("b.pdf", 3), ("c.pdf", 1)):
    escritor = PdfWriter()
    for _ in range(paginas):
        escritor.add_blank_page(595.28, 841.89)
    escritor.write(os.path.join("pdfs", nombre))

def elimina_ultima_pagina(origen, destino):
    lector, escritor = PdfReader(origen), PdfWriter()
    for pagina in lector.pages[:-1]:
        escritor.add_page(pagina)
    escritor.write(destino)

def une_carpeta(carpeta, destino):
    escritor = PdfWriter()
    for nombre in sorted(os.listdir(carpeta)):
        if nombre.endswith(".pdf"):
            escritor.append(os.path.join(carpeta, nombre))
    escritor.write(destino)

elimina_ultima_pagina("pdfs/b.pdf", "b_recortado.pdf")
une_carpeta("pdfs", "unidos.pdf")
print(len(PdfReader("b_recortado.pdf").pages), len(PdfReader("unidos.pdf").pages))

2 6


## 4. Porcentajes en Excel con `openpyxl`

Crear un Excel con los gastos y su total, y añadir la columna *Porcentaje (%)* con una fórmula que calcule el peso de cada gasto sobre el total (el alquiler, 800 €, es el 61 % de 1310 €).

In [4]:
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font

datos = [["Alquiler", 800], ["Electricidad", 100], ["Agua", 50], ["Internet", 60], ["Supermercado", 300]]
wb = Workbook()
ws = wb.active
ws.append(["Concepto", "Gasto (€)", "Porcentaje (%)"])
for celda in ws[1]:
    celda.font = Font(bold=True)
ultima = len(datos) + 1
total = ultima + 1
for i, (concepto, gasto) in enumerate(datos, start=2):
    ws.append([concepto, gasto, f"=B{i}/B${total}*100"])
    ws[f"C{i}"].number_format = "0.0"
ws.append(["Gastos Totales", f"=SUM(B2:B{ultima})", f"=SUM(C2:C{ultima})"])
wb.save("gastos.xlsx")
print([[c.value for c in fila] for fila in load_workbook("gastos.xlsx").active.iter_rows()])
print({c: round(g / 1310 * 100, 1) for c, g in datos})

[['Concepto', 'Gasto (€)', 'Porcentaje (%)'], ['Alquiler', 800, '=B2/B$7*100'], ['Electricidad', 100, '=B3/B$7*100'], ['Agua', 50, '=B4/B$7*100'], ['Internet', 60, '=B5/B$7*100'], ['Supermercado', 300, '=B6/B$7*100'], ['Gastos Totales', '=SUM(B2:B6)', '=SUM(C2:C6)']]
{'Alquiler': 61.1, 'Electricidad': 7.6, 'Agua': 3.8, 'Internet': 4.6, 'Supermercado': 22.9}


## 5. Proyecto con entorno virtual

Crear un directorio de proyecto con su propio entorno virtual (como el `.venv` de este repositorio). Desde la terminal sería `python -m venv .venv` y activarlo con `.venv\Scripts\activate` (Windows) o `source .venv/bin/activate` (Linux/macOS).

In [5]:
import venv

venv.create("mi_proyecto/.venv", with_pip=False)
with open("mi_proyecto/requirements.txt", "w") as f:
    f.write("flask\n")
print(sorted(os.listdir("mi_proyecto")), sorted(os.listdir("mi_proyecto/.venv")))
print(open("mi_proyecto/.venv/pyvenv.cfg").read().splitlines()[0])

['.venv', 'requirements.txt'] ['Include', 'Lib', 'Scripts', 'pyvenv.cfg']
home = C:\Users\hecto\AppData\Local\Programs\Python\Python313
